# 第 4 章：TD 学习 —— 从样本中学习

上一章的 DP 有个让人不放心的前提：**P 和 R 你全知道**。可真实世界谁来告诉你转移概率？学开车的乘客不会给你一张「踩油门 0.3 秒后速度分布表」——你只能**自己开、自己感受、自己总结**。这一章，agent 终于走进现实：不再有模型，只有一条条真实经历过的轨迹样本。

从样本里估计期望，最朴素的想法是蒙特卡洛（MC）：把回报平均一下就行。但 MC 有个致命的别扭之处——**必须等整局结束**才能更新。下了一半的棋，前面那步的好坏要等终局才知道？我们将在本章造出 RL 最重要的一个思想：**TD（时序差分）**——不等终局，每走一步就用「下一步的猜测」修正「这一步的猜测」。你在生活里早就在用它：考试交卷前检查出一道错题时，你会立刻修正对「这类题我会不会」的判断——而不是等到出分那天。

> 🌍 **真实世界**：TD 的第一个成名作是 1992 年的 TD-Gammon——一个**从零开始**、只靠自我对弈的输赢信号学会西洋双陆棋的程序，棋力达到人类世界冠军水平。它是 AlphaGo 的精神祖先，证明了「不用人类棋谱、只靠 TD + 自我对弈」这条路走得通。

## 学习目标

1. 理解 **蒙特卡洛（MC）** 和 **TD(0)** 的本质区别
2. 推导 TD(0) 的更新规则与**期望收敛性**
3. 复现 Sutton-Barto 的经典 **Random Walk** 实验
4. 掌握 **n-step TD** 和 **TD(λ)**（eligibility traces）
5. 直观理解 **bias/variance tradeoff**

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from utils import set_seed, make_interactive, plot_bar_compare
from rlenvs import RandomWalk, GridWorld, small_grid_5x5

set_seed(0)

## 4.1 放弃"知道 P 和 R"的假设

Ch03 的 DP 算法用到了：

- $P(s'|s,a)$：从 $s$ 选 $a$ 到 $s'$ 的概率
- $R(s,a)$：奖励函数

但现实中你**不知道**这些。比如开车时：
- 你不知道前面那辆车下一秒会怎么开
- 你不知道某个路况会不会打滑

**Model-free RL** 只用采到的轨迹 $(S_0, A_0, R_1, S_1, A_1, R_2, \dots)$ 学。

## 4.2 蒙特卡洛（MC）估计

最直接的想法：**多次采样，求平均**。

### First-visit MC

对每个状态 $s$，记录所有 "首次访问 $s$" 之后的回报 $G_t$，取平均作为 $V(s)$。

### Every-visit MC

不要求"首次"，每次访问 $s$ 都记录 $G_t$。

两者在 episode 长、状态访问次数有限时**有差异**，但都收敛到 $V^\pi$。

### MC 的特点

- **无偏**：估计的 $V$ 不带系统性偏差
- **高方差**：每次 $G_t$ 是从 $t$ 到终止的累计奖励，方差大
- **不需要 bootstrap**：不依赖其他 $V$ 估计
- **必须等 episode 结束**：无法在线学习

In [ ]:
def run_episode_uniform(env_grid, start_state, gamma=1.0, max_steps=500):
    """在 GridWorld 上从 start_state 出发，用均匀随机策略走完一个 episode。"""
    env_grid._state = start_state
    trajectory = []
    done = False
    t = 0
    s = start_state
    while not done and t < max_steps:
        a = np.random.randint(env_grid.nA)
        s_next, r, done, _ = env_grid.step(a)
        trajectory.append((s, a, r, s_next))
        s = s_next
        t += 1
    return trajectory


def mc_estimate_first_visit(env_grid, gamma=0.9, n_episodes=5000):
    """first-visit MC，估计 V^π（π 是均匀随机策略）。"""
    nS = env_grid.nS
    returns_sum = np.zeros(nS)
    returns_cnt = np.zeros(nS)
    for _ in range(n_episodes):
        start = env_grid.reset()
        traj = run_episode_uniform(env_grid, start, gamma=gamma)
        # 反向累计 G，每步存 (s, G_t)
        G = 0.0
        Gs_rev = []
        for s, a, r, s_next in reversed(traj):
            G = r + gamma * G
            Gs_rev.append((s, G))
        # Gs_rev 是反向的，转回来
        Gs = list(reversed(Gs_rev))
        # first-visit：只取每个 s 第一次出现的 G
        seen = set()
        for s, G in Gs:
            if s in seen:
                continue
            seen.add(s)
            returns_sum[s] += G
            returns_cnt[s] += 1
    V = np.where(returns_cnt > 0, returns_sum / np.maximum(returns_cnt, 1), 0.0)
    return V, returns_cnt


env = small_grid_5x5(seed=0)
V_mc, cnt = mc_estimate_first_visit(env, gamma=0.9, n_episodes=3000)
print(f"MC 估计的 V^π（用了 {cnt.sum()} 个 first-visit 样本）")
print(V_mc.reshape(env.shape).round(2))

## 4.3 TD(0)：把 MC 和 DP 结合起来

TD(0) 的洞察：**不必等 episode 结束**。每一步都用 **当前一步奖励 + 对下一步的估计** 来更新。

### TD(0) 更新规则

$$
V(S_t) \leftarrow V(S_t) + \alpha \big[ \underbrace{R_{t+1} + \gamma V(S_{t+1})}_{\text{TD target}} - V(S_t) \big]
$$

记号：

- **TD target**：$R_{t+1} + \gamma V(S_{t+1})$，新的 $G_t$ 估计
- **TD error**：$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$，更新方向

直觉：**朝 TD target 走一小步**。

### 和 MC、DP 的关系

| | MC | DP | TD(0) |
|---|---|---|---|
| 用样本？ | ✓ | ✗ | ✓ |
| 用 bootstrap（依赖其他 V）？ | ✗ | ✓ | ✓ |

TD(0) 是两者的混血：**采样** + **bootstrap**。

### TD(0) 的特点

- **有偏**：因为 target 用了其他 $V$ 估计（这些估计不准）
- **低方差**：只看一步奖励
- **在线**：每步都能更新，不用等 episode 结束
- **在 continuing task（无终止）也能用**

## 4.4 TD(0) 的收敛性

<details>
<summary><b>📝 TD(0) 在期望下的等价更新（点开看）</b></summary>

考虑 $\alpha \to 0$、无穷多样本的极限。对 $V(S_t)$ 的更新量为：

$$
\mathbb{E}[\delta_t | S_t = s] = \mathbb{E}[R_{t+1} + \gamma V(S_{t+1}) - V(s) | S_t = s]
$$

$$
= \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) [r(s,a,s') + \gamma V(s')] - V(s)
$$

$$
= (T^\pi V)(s) - V(s)
$$

其中 $T^\pi$ 是贝尔曼 backup 算子。当 $V = V^\pi$ 时 $(T^\pi V)(s) = V(s)$，所以 $V^\pi$ 是这个动力系统的不动点。

由 $T^\pi$ 的 $\gamma$-压缩性（$\|T^\pi V_1 - T^\pi V_2\| \leq \gamma \|V_1 - V_2\|$），TD(0) 在 step-size 满足 Robbins-Monro 条件（$\sum \alpha = \infty, \sum \alpha^2 < \infty$）时几乎必然收敛到 $V^\pi$。

</details>

**直觉**：TD(0) 是用采样实现的 Ch03 迭代策略评估。

In [ ]:
def td0_estimate(env_grid, gamma=0.9, alpha=0.1, n_episodes=5000, max_steps=200):
    """TD(0) 估计 V^π（π 是均匀随机）。"""
    nS = env_grid.nS
    V = np.zeros(nS)
    for _ in range(n_episodes):
        s = env_grid.reset()
        done = False
        t = 0
        while not done and t < max_steps:
            a = np.random.randint(env_grid.nA)
            s_next, r, done, _ = env_grid.step(a)
            # TD(0) 更新
            td_target = r + gamma * (0.0 if done else V[s_next])
            V[s] += alpha * (td_target - V[s])
            s = s_next
            t += 1
    return V


V_td = td0_estimate(env, gamma=0.9, alpha=0.05, n_episodes=5000)

# 用 DP 算精确解对比
pi_uniform = np.full((env.nS, env.nA), 1.0 / env.nA)
R_pi = (pi_uniform * env.R).sum(axis=1)
P_pi = np.einsum('sa,saq->sq', pi_uniform, env.P)
V_exact = np.linalg.solve(np.eye(env.nS) - 0.9 * P_pi, R_pi)

print(f"{'state':<6}{'V_exact':<12}{'V_td0':<12}{'V_mc':<12}")
for s in [0, 5, 10, 15, 20, 24]:
    print(f"{s:<6}{V_exact[s]:<12.3f}{V_td[s]:<12.3f}{V_mc[s]:<12.3f}")

print(f"\nTD(0) vs exact 最大误差: {np.abs(V_td - V_exact).max():.3f}")
print(f"MC vs exact 最大误差: {np.abs(V_mc - V_exact).max():.3f}")

## 4.5 经典实验：Sutton-Barto Random Walk

**这是 RL 教材最经典的图之一**——Sutton & Barto 教科书图 6.2 的复现，几乎每门 RL 课都会让学生亲手跑一遍。

> 🤔 **先猜再跑**：下面的实验跑 α = 0.05/0.10/0.15/0.20 四档的 TD(0)，画「各状态的 V 估计随 episode 变化」的折线，真值是灰色横线。预测：**episode 数很少时（比如 3 个），哪一档 α 的估计离真值最远？猜一个方向（整体偏高/偏低/乱跳）再跑。**
>
> <details><summary>写下猜测再点开</summary>
>
> 提示：TD target 里有 V(s')——初期 V 全是 0，target 系统性偏小，估计**从下往上爬**。α 越大爬得越快、但每一步都被单条轨迹的噪声拽得越狠。你会同时看到 bias 的「爬升」和 variance 的「毛刺」——这张图就是 §4.6 bias/variance 分析的实物版。
> </details>

Sutton-Barto 19 状态随机游走：

- 19 个状态排一条直线（编号 1..19）
- 从中间 (state 10) 出发
- 每步 50% 往左、50% 往右
- 落到 state 0（左端）奖励 -1
- 落到 state 20（右端）奖励 +1

真实 $V^\pi$ 是线性插值：$V(s) = -1 + 2s/20$，从 -0.9 到 +0.9。

我们看 TD(0) 在不同 $\alpha$ 下能多快收敛到真值。

In [ ]:
env_rw = RandomWalk(n_states=19, left_reward=-1.0, right_reward=1.0, seed=0)
true_V = env_rw.true_values()
print(f"真实 V^π：{true_V.round(2)}")

def td0_random_walk(env_rw, alpha=0.1, n_episodes=10):
    """TD(0) 在 RandomWalk 上估计 V^π。"""
    V = np.zeros(env_rw.nS)  # 内部状态 1..n_states
    V_history = [V.copy()]
    for ep in range(n_episodes):
        s = env_rw.reset()  # 返回 1..n_states
        done = False
        while not done:
            s_next, r, done, _ = env_rw.step()
            # 转回内部 0-indexed
            s_idx = s - 1
            s_next_idx = s_next - 1 if s_next != 0 and s_next != env_rw.nS + 1 else 0
            td_target = r if done else r + 0.9 * V[s_next_idx]
            V[s_idx] += alpha * (td_target - V[s_idx])
            s = s_next
        V_history.append(V.copy())
    return V, V_history


# 在前几次 episode 的几个 α 下画 V
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
alphas = [0.05, 0.1, 0.15]
episodes_to_show = [0, 1, 10, 50, 100]

for col, alpha in enumerate(alphas):
    np.random.seed(0)
    V, hist = td0_random_walk(env_rw, alpha=alpha, n_episodes=100)
    for row, ep_idx in enumerate([1, 100]):
        ax = axes[row][col]
        x = np.arange(1, 20)
        ax.plot(x, true_V, 'k-', linewidth=2, label='true $V^\pi$')
        ax.plot(x, hist[min(ep_idx, len(hist)-1)], 'r.--', linewidth=1.5, label=f'TD est (ep={ep_idx})')
        ax.set_title(f'α={alpha}, episode={ep_idx}')
        ax.set_xlabel('state')
        ax.set_ylabel('V')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

你应该看到：

- **Episode 1 后**：TD 估计只在中间几个状态有非零值（因为只走过那些）
- **Episode 100 后**：TD 几乎完全对齐真值
- **α 越大收敛越快**，但太大会震荡

## 4.6 MC vs TD 的 bias/variance

我们做一个对比实验：在 RandomWalk 上，比较 MC（first-visit）和 TD(0) 估计的**偏差**（系统性偏离真值）和**方差**。

In [ ]:
def mc_random_walk(env_rw, gamma=1.0, n_episodes=100, alpha=0.01):
    """first-visit MC，每步用 alpha 增量更新。"""
    V = np.zeros(env_rw.nS)
    for ep in range(n_episodes):
        # 跑一个 episode
        s = env_rw.reset()
        trajectory = []
        done = False
        while not done:
            s_next, r, done, _ = env_rw.step()
            trajectory.append((s, r))
            s = s_next
        # 算 G_t 反向
        G = 0.0
        visited_G = {}
        for s, r in reversed(trajectory):
            G = r + gamma * G
            visited_G[s] = G
        for s, G in visited_G.items():
            V[s - 1] += alpha * (G - V[s - 1])
    return V


# 跑 100 次实验，统计 bias 和 variance
n_runs = 100
n_eps = 100
V_td_runs = np.zeros((n_runs, 19))
V_mc_runs = np.zeros((n_runs, 19))
for run in range(n_runs):
    np.random.seed(run)
    env_rw = RandomWalk(n_states=19, seed=run)
    V_td_runs[run] = td0_random_walk(env_rw, alpha=0.1, n_episodes=n_eps)[0]
    V_mc_runs[run] = mc_random_walk(env_rw, n_episodes=n_eps, alpha=0.01)

td_bias = (V_td_runs.mean(axis=0) - true_V).mean()
mc_bias = (V_mc_runs.mean(axis=0) - true_V).mean()
td_var = V_td_runs.var(axis=0).mean()
mc_var = V_mc_runs.var(axis=0).mean()

print(f"{'':<10}{'TD(0)':<12}{'MC':<12}")
print(f"{'bias':<10}{td_bias:<12.4f}{mc_bias:<12.4f}")
print(f"{'variance':<10}{td_var:<12.4f}{mc_var:<12.4f}")

# 画一个柱状图
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_bar_compare([abs(td_bias), abs(mc_bias)], ['TD(0)', 'MC'],
                 colors=['steelblue', 'crimson'], title='|bias|', ax=axes[0])
plot_bar_compare([td_var, mc_var], ['TD(0)', 'MC'],
                 colors=['steelblue', 'crimson'], title='variance', ax=axes[1])
plt.tight_layout(); plt.show()

你应该看到：

- **TD(0) 偏差更大**（因为 bootstrap 引入偏差）
- **MC 方差更大**（因为累计奖励方差叠加）
- 这是经典的 **bias-variance tradeoff**

## 4.7 n-step TD：MC 和 TD(0) 之间

TD(0) 只看 1 步。MC 看到终止。**n-step TD** 看中间任意 $n$ 步：

$$
G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + \dots + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})
$$

直觉：

- $n = 1$：TD(0)，**低方差、高偏差**
- $n = \infty$（或到终止）：MC，**高方差、无偏差**
- $n$ 中间：折中

In [ ]:
def n_step_td_random_walk(env_rw, n=3, alpha=0.1, gamma=1.0, n_episodes=100):
    """n-step TD on RandomWalk。"""
    V = np.zeros(env_rw.nS)
    for ep in range(n_episodes):
        s = env_rw.reset()
        # 缓存 rewards 和 states
        states = [s]
        rewards = [0]  # 占位，让索引对齐
        T = float('inf')
        t = 0
        while True:
            if t < T:
                s_next, r, done, _ = env_rw.step()
                states.append(s_next)
                rewards.append(r)
                if done:
                    T = t + 1
            tau = t - n + 1  # 现在 update 的时刻
            if tau >= 0:
                # G_tau^{(n)}
                G = 0.0
                upper = min(tau + n, T)
                for k in range(tau + 1, upper + 1):
                    G += gamma ** (k - tau - 1) * rewards[k]
                if tau + n < T:
                    s_end_idx = states[tau + n] - 1
                    if 0 <= s_end_idx < env_rw.nS:
                        G += gamma ** n * V[s_end_idx]
                s_tau_idx = states[tau] - 1
                if 0 <= s_tau_idx < env_rw.nS:
                    V[s_tau_idx] += alpha * (G - V[s_tau_idx])
            t += 1
            if tau == T - 1:
                break
            if t > 1000:  # 安全 break
                break
    return V


# 比较 n=1, 3, ∞（MC）
n_steps_to_test = [1, 2, 3, 5, 10, 30]
n_runs = 30
rmses = []
for n in n_steps_to_test:
    errs = []
    for run in range(n_runs):
        np.random.seed(run)
        env_rw = RandomWalk(n_states=19, seed=run)
        V = n_step_td_random_walk(env_rw, n=n, alpha=0.1 if n <= 5 else 0.05, n_episodes=10)
        err = np.sqrt(np.mean((V - true_V) ** 2))
        errs.append(err)
    rmses.append(np.mean(errs))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_steps_to_test, rmses, 'o-', linewidth=2, markersize=10)
ax.set_xlabel('n (steps)')
ax.set_ylabel('RMS error over 10 episodes')
ax.set_title('n-step TD：n 越大方差越大')
ax.set_xscale('log')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4.8 TD(λ)：用 eligibility traces 平滑所有 n

$n$-step TD 让你**选一个 n**。但如果我想**同时用所有 n 的平均**呢？

**TD(λ)** 用 backward view（eligibility traces）实现这个想法：

### Eligibility trace

每个状态 $s$ 有一个 trace $e_t(s)$，每步更新：

$$
e_t(s) = \gamma \lambda e_{t-1}(s) + \mathbb{1}[S_t = s]
$$

- $\lambda = 0$：只有当前状态有 trace，等价于 TD(0)
- $\lambda = 1$：trace 不衰减，等价于 MC（在 episodic 任务里）
- $\lambda \in (0, 1)$：在两者之间

### TD(λ) 更新

每步用**同一个 TD error** 更新**所有状态**，按它们的 trace 加权：

$$
\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)
$$

$$
V(s) \leftarrow V(s) + \alpha \delta_t e_t(s), \quad \forall s
$$

直觉：最近访问过的状态、且权重没衰减太多（$\gamma \lambda$）的状态，应该承担这次误差更多。

In [ ]:
def td_lambda_random_walk(env_rw, lam=0.5, alpha=0.1, gamma=1.0, n_episodes=100):
    """TD(λ) with eligibility traces on RandomWalk。"""
    V = np.zeros(env_rw.nS)
    for ep in range(n_episodes):
        e = np.zeros(env_rw.nS)  # eligibility trace
        s = env_rw.reset()
        done = False
        while not done:
            s_next, r, done, _ = env_rw.step()
            s_idx = s - 1
            s_next_idx = max(0, min(env_rw.nS - 1, s_next - 1))
            td_target = r if done else r + gamma * V[s_next_idx]
            delta = td_target - V[s_idx]
            # 更新 trace：所有状态的 trace 都衰减
            e = gamma * lam * e
            e[s_idx] += 1.0
            # 用同一个 delta 更新所有状态
            V += alpha * delta * e
            s = s_next
    return V


# 比较 λ = 0, 0.3, 0.5, 0.8, 1
lambdas = [0.0, 0.3, 0.5, 0.8, 0.9, 1.0]
n_runs = 30
rmses = []
for lam in lambdas:
    errs = []
    for run in range(n_runs):
        np.random.seed(run)
        env_rw = RandomWalk(n_states=19, seed=run)
        V = td_lambda_random_walk(env_rw, lam=lam, alpha=0.1, n_episodes=10)
        err = np.sqrt(np.mean((V - true_V) ** 2))
        errs.append(err)
    rmses.append(np.mean(errs))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lambdas, rmses, 's-', linewidth=2, markersize=10, color='purple')
ax.set_xlabel('λ')
ax.set_ylabel('RMS error over 10 episodes')
ax.set_title('TD(λ)：中间的 λ 往往最好')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**典型结果**：$\lambda \approx 0.5$ 附近最好——既不太偏、又不太噪。

## 4.9 交互式 widget：调 $\alpha$ 和 $\lambda$

In [ ]:
def td_lambda_demo(alpha=0.1, lam=0.5):
    np.random.seed(0)
    env_rw = RandomWalk(n_states=19, seed=0)
    V = td_lambda_random_walk(env_rw, lam=lam, alpha=alpha, n_episodes=20)
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(1, 20)
    ax.plot(x, true_V, 'k-', linewidth=2, label='true $V^\pi$')
    ax.plot(x, V, 'r.--', linewidth=1.5, label=f'TD(λ) est')
    ax.set_title(f'α={alpha}, λ={lam}, 20 episodes')
    ax.set_xlabel('state'); ax.set_ylabel('V')
    ax.legend(); ax.grid(alpha=0.3); ax.set_ylim(-1.1, 1.1)
    plt.tight_layout(); plt.show()

w = make_interactive(td_lambda_demo,
                     params={'alpha': (0.1, 0.01, 0.5, 0.01),
                             'lam':   (0.5, 0.0, 1.0, 0.05)},
                     layout='hbox')

## 4.10 小结

| 方法 | 看几步 | 偏差 | 方差 | 在线 |
|---|---|---|---|---|
| MC | 全部 | 无 | 高 | ✗（等 episode 结束） |
| TD(0) | 1 步 | 有 | 低 | ✓ |
| n-step | n 步 | 中 | 中 | ✓ |
| TD(λ) | 全部加权（按 λ） | 可调 | 可调 | ✓ |

**核心收获**：

1. **bootstrap**（用估计更新估计）是 RL 的关键设计选择
2. TD 把 MC 和 DP 的优势结合：**采样** + **bootstrap**
3. **bias-variance tradeoff** 是 RL 调参的核心
4. **TD(λ) 的 eligibility trace 思想**会在后面的 Actor-Critic、PPO 里反复出现

## 4.11 📝 练习

### 练习 1：在 GridWorld 上比较 MC vs TD(0)

用 `small_grid_5x5`，对比 MC 和 TD(0) 估计的 $V^\pi$：

1. 固定 $\gamma = 0.9$，跑 100 个 seed
2. 画两条 RMS-error-vs-episodes 曲线
3. 哪个收敛更快？为什么？

### 练习 2：online TD(λ) for control

把本章的 TD(λ) 推广到 **SARSA(λ)**：用 $Q$ 替代 $V$，结合 Ch05 的 SARSA 思想，在 GridWorld 上学习 $\pi^*$。

**提示**：eligibility trace 从 $e(s)$ 变成 $e(s, a)$——每次 step 后全体 `e *= γλ`、刚访问的 `(s, a)` 加 1；更新式 `Q += α · δ · e`，其中 δ 用 SARSA 的 target（$r + \gamma Q(s', a') - Q(s, a)$，$a'$ 是实际采样的下一个动作）。

> 参考答案：练习 1 → `solutions/ch04_mc_vs_td_gridworld.ipynb`；练习 2 是开放练习（无参考答案）——写完后可以和 Ch05 的 Q-learning 对比收敛速度。
>
> 📖 两章都完成后，做 `STUDY_GUIDE.md` 里 Ch04 的自测题（4 题）。

---

下一章：**第 5 章 — Q-learning 和 SARSA**。
我们将从 **预测** 跳到 **控制**——学习最优策略 $\pi^*$，并区分 **on-policy vs off-policy** 这个 PPO/GRPO 都绕不开的核心概念。